# Tamil call transcription — Kaggle 2x T4

Before running:
1. **Accelerator**: Settings -> Accelerator -> **GPU T4 x2**.
2. **Attach your audio datasets** (+ Add Data). The script scans all of
   `/kaggle/input` recursively — every folder there is searched for audio.
3. **Resume across sessions**: after each session, save `/kaggle/working/out`
   as a new Kaggle dataset (or a new version of an existing one). Next
   session, attach that dataset too and copy it into `/kaggle/working/out`
   (cell below) — the manifest skips finished files.
4. If this repo is private, attach the `src/` folder as a Kaggle dataset
   instead of cloning from GitHub in the cell below.

In [ ]:
!nvidia-smi -L
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip -q install faster-whisper ctranslate2 transformers huggingface_hub av

In [ ]:
import os

WORK = '/kaggle/working'
OUT = f'{WORK}/out'
MODEL = f'{WORK}/tamil-whisper-ct2'
REPO = f'{WORK}/voice_processing'

# Restore previous session's outputs so the run resumes.
# Change this to the slug of the dataset you saved last session.
PREV_OUT = '/kaggle/input/tamil-transcripts-out'  # e.g. /kaggle/input/<your-dataset>
if os.path.isdir(PREV_OUT):
    !cp -r "$PREV_OUT" "$OUT"
    print('restored previous outputs')
else:
    print('no previous output dataset attached — starting fresh')

In [ ]:
# Get the pipeline code. Public repo -> clone. Private -> attach src/ as a
# Kaggle dataset and point REPO at it instead.
if not os.path.isdir(REPO):
    !git -C "$WORK" clone -q https://github.com/srikrishnadeveloper/voice_processing.git
!ls "$REPO/src"

In [ ]:
# One-time conversion of the HF checkpoint to CTranslate2 (int8_float16).
# ~3 GB download + a few minutes. Skipped automatically if MODEL exists
# (save it into your output dataset to skip this in later sessions).
if not os.path.isdir(MODEL):
    !ct2-transformers-converter \
        --model vasista22/whisper-tamil-large-v2 \
        --output_dir "$MODEL" \
        --copy_files tokenizer.json preprocessor_config.json \
        --quantization int8_float16
else:
    print('model already converted')

In [ ]:
# EVAL FIRST: transcribe 4 files on one GPU and eyeball quality before
# committing GPU hours to the full corpus.
!python "$REPO/src/transcribe.py" \
    --input-dir /kaggle/input \
    --output-dir "$OUT" \
    --model "$MODEL" \
    --language ta \
    --workers 1 \
    --max-files 4

In [ ]:
import glob

for txt in glob.glob(f'{OUT}/transcripts/*.txt')[:4]:
    print('=' * 60)
    print(txt)
    print('-' * 60)
    print(open(txt, encoding='utf-8').read()[:2000])

## Full run

If the eval transcripts look right, run the full batch. `--max-minutes`
stops scheduling new work before Kaggle's 12 h session cap; files already
finished stay saved. Re-run next session after saving `out/` as a dataset.

To try the faster fallback model instead, convert
`deepdml/whisper-large-v3-turbo-ta-mix-norm` the same way and point
`--model` at it.

In [ ]:
!python "$REPO/src/transcribe.py" \
    --input-dir /kaggle/input \
    --output-dir "$OUT" \
    --model "$MODEL" \
    --language ta \
    --workers 2 \
    --batch-size 16 \
    --compute-type int8_float16 \
    --max-minutes 640

In [ ]:
# Progress summary across all manifest shards.
import glob, json

rows = []
for m in glob.glob(f'{OUT}/manifest*.jsonl'):
    rows += [json.loads(l) for l in open(m, encoding='utf-8') if l.strip()]
ok = [r for r in rows if r.get('status') == 'ok']
err = [r for r in rows if r.get('status') == 'error']
mins = sum(r.get('duration_sec', 0) for r in ok) / 60
print(f'transcribed={len(ok)} errors={len(err)} audio_minutes={mins:.0f}')
for r in err[:20]:
    print('ERR', r['file'], r.get('error'))

## Persist before the session ends

Save `/kaggle/working/out` (and optionally the converted `tamil-whisper-ct2`
model dir) as a Kaggle dataset: notebook **Output** panel -> **New Dataset**,
or version an existing dataset. Next session, attach it and set `PREV_OUT`
above — the manifest makes the run resume where it stopped.

**Privacy:** transcripts contain personal information — keep the output
dataset private.